# 🚀 Eksperimen Alternatif 2: Validation-Based Dynamic Threshold Tuning vs Default 0.5
## Optimalisasi Deteksi Flaky Test pada Skenario Ekstrem Imbalanced Cross-Project

- **Kelompok Peneliti**: Kelompok A01 — Rekayasa Perangkat Lunak A (Magister Teknik Informatika ITS)
- **Hardware Akselerasi**: Laptop GPU NVIDIA GeForce RTX 4050 (6GB GDDR6 VRAM, CUDA 13.4, Driver 617.42)
- **Dataset**: FlakeFlagger (22.236 test cases dari 24 repositori Java, C1 Row-Level Drop: 21.963 baris)
- **Paper Rujukan**: 
  1. Alshammari et al. (ICSE 2021 — *FlakeFlagger: Predicting Flakiness Without Rerunning Tests*)
  2. Afeltra et al. (IEEE Access 2024 — *A Large-Scale Empirical Investigation Into Cross-Project Flaky Test Prediction*)
  3. Provost (AAAI 2000 — *Machine Learning from Imbalanced Data Sets: Foundational Issues and Threshold Moving*)

### 1. Verifikasi Lingkungan GPU NVIDIA RTX 4050 & CUDA 13

In [ ]:
import sys
from pathlib import Path

# Tambahkan path src ke sys.path
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from check_gpu import check_nvidia_smi, check_xgboost_gpu
check_nvidia_smi()
check_xgboost_gpu()

### 2. Pemuatan & Pemeriksaan Dataset FlakeFlagger (C1: Row-Level Drop)

In [ ]:
from data_loader import load_dataset

df, features, projects = load_dataset("C1")
print(f"Total baris: {len(df):,}")
print(f"Total fitur prediktif: {len(features)}")
print(f"Jumlah proyek: {len(projects)}")
print(f"Total flaky tests: {df['flaky'].sum()} ({df['flaky'].mean()*100:.2f}%)")

### 3. Eksekusi LOPO-CV untuk 5 Strategi Dynamic Threshold Tuning (Zero Test Leakage)

In [ ]:
from lopo_threshold_runner import run_lopo_dynamic_threshold

df_folds, df_summary = run_lopo_dynamic_threshold(
    config_path="../configs/threshold_config.json",
    models_to_run=["xgboost_gpu", "random_forest"]
)

display(df_summary)

### 4. Uji Signifikansi Statistik (Wilcoxon Signed-Rank Test & Cliff's Delta)

In [ ]:
from statistical_tests import run_wilcoxon_analysis

df_wilcoxon = run_wilcoxon_analysis(
    folds_csv_path="../results/fold_thresholds.csv",
    output_file="../results/wilcoxon_test_results.csv",
    model_name="xgboost_gpu"
)

display(df_wilcoxon[df_wilcoxon["Metric"].isin(["F1", "Recall", "Precision"])])

### 5. Pembuatan dan Tampilan Visualisasi Publikasi Ilmiah

In [ ]:
from visualizer import generate_all_plots
from IPython.display import Image, display

generate_all_plots(
    folds_csv_path="../results/fold_thresholds.csv",
    summary_csv_path="../results/threshold_comparison_summary.csv",
    plots_dir="../results/plots"
)

display(Image(filename="../results/plots/boxplot_f1_strategies_xgboost_gpu.png"))
display(Image(filename="../results/plots/boxplot_precision_recall_tradeoff_xgboost_gpu.png"))
display(Image(filename="../results/plots/distribution_learned_thresholds_xgboost_gpu.png"))
display(Image(filename="../results/plots/barchart_metrics_comparison_xgboost_gpu.png"))
display(Image(filename="../results/plots/per_project_f1_gain_xgboost_gpu.png"))